# Sentiment Analysis — Amazon Product Reviews

**Disciplina:** Engenharia de Software para IA e Frameworks Profundos  
**Dataset:** Amazon Product Reviews (Kaggle: yasserh)

---
> **Como usar:** Execute **Kernel → Restart & Run All** para rodar todas as entregas de uma vez.

In [40]:
import sys, os
from pathlib import Path

# VS Code injeta __vsc_ipynb_file__ com o caminho absoluto do notebook.
# Isso resolve o cwd incorreto independente de onde o Jupyter foi aberto.
if "__vsc_ipynb_file__" in dir():
    ROOT = str(Path(__vsc_ipynb_file__).parent.parent)
else:
    # Fallback para Jupyter clássico: sobe na árvore procurando main.py
    for p in [Path(os.getcwd())] + list(Path(os.getcwd()).parents):
        if (p / "main.py").exists() and (p / "requirements.txt").exists():
            ROOT = str(p)
            break
    else:
        raise RuntimeError(
            "Raiz do projeto não encontrada.\n"
            "Abra o notebook pelo VS Code ou execute o Jupyter de dentro da pasta do projeto."
        )

os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"ROOT  : {ROOT}")
print(f"CWD   : {os.getcwd()}")
assert os.path.exists("data/raw/reviews.csv"), "ERRO: data/raw/reviews.csv não encontrado!"
print("Dataset: OK")

ROOT  : c:\Users\cliberato\Desktop\UFPE\Engenharia de Software\project-sentiment-analysis
CWD   : c:\Users\cliberato\Desktop\UFPE\Engenharia de Software\project-sentiment-analysis
Dataset: OK


---
## Entrega 1 — Funções, Modularização e Tipagem
> Etapas 1, 2 e 3

Código organizado em módulos com responsabilidades separadas. Todas as funções têm type hints.

```
src/
├── data/loader.py              carregamento e validação
├── preprocessing/transform.py  limpeza de texto e labels
├── models/model.py             definição do modelo
├── training/train.py           pipeline de treinamento
├── evaluation/metrics.py       métricas de avaliação
├── inference/predict.py        inferência em produção
└── utils/config.py             constantes e hiperparâmetros
```

In [41]:
import inspect
from src.data.loader import load_data, validate_columns, inspect_data
from src.preprocessing.transform import clean_text, normalize_label, preprocess_dataset

print("=== Assinaturas tipadas das funções públicas ===")
for fn in [load_data, validate_columns, inspect_data, clean_text, normalize_label, preprocess_dataset]:
    print(f"  def {fn.__name__}{inspect.signature(fn)}")

print("\n=== Demonstração: clean_text e normalize_label ===")
casos = [("This is GREAT!!!", 5), ("Terrible product...", 1), ("It's okay.", 3)]
print(f"{'Texto original':<28} {'clean_text':<28} {'label'}")
print("-" * 68)
for texto, rating in casos:
    print(f"{texto:<28} {clean_text(texto):<28} {normalize_label(rating)}")

=== Assinaturas tipadas das funções públicas ===
  def load_data(path: str) -> pandas.core.frame.DataFrame
  def validate_columns(df: pandas.core.frame.DataFrame, required: list[str]) -> None
  def inspect_data(df: pandas.core.frame.DataFrame) -> None
  def clean_text(text: 'str') -> 'str'
  def normalize_label(rating: 'int') -> 'int | None'
  def preprocess_dataset(df: 'pd.DataFrame') -> 'pd.DataFrame'

=== Demonstração: clean_text e normalize_label ===
Texto original               clean_text                   label
--------------------------------------------------------------------
This is GREAT!!!             this is great                1
Terrible product...          terrible product             0
It's okay.                   its okay                     None


---
## Entrega 2 — NumPy
> Etapa 4

- `inspect_data` → `np.min`, `np.max`, `np.mean`, `np.std`, `np.unique`
- `vectorize_texts` → TF-IDF `.toarray()` → matriz densa NumPy
- `normalize_features` → normalização L2 com `np.linalg.norm`

In [42]:
import numpy as np
from src.data.loader import load_data, inspect_data
from src.preprocessing.transform import preprocess_dataset, vectorize_texts, normalize_features
from src.training.train import split_dataset
from src.utils.config import DATA_PATH, TEXT_COLUMN, LABEL_COLUMN, TEST_SIZE, MAX_FEATURES

df = load_data(DATA_PATH)
print("=== Análise NumPy do dataset ===")
inspect_data(df)

df_clean = preprocess_dataset(df)
print(f"\nAmostras após pré-processamento: {len(df_clean)}")
print(f"Distribuição:\n{df_clean[LABEL_COLUMN].value_counts().to_string()}")

x_train, x_test, y_train, y_test = split_dataset(df_clean, TEXT_COLUMN, LABEL_COLUMN, TEST_SIZE)

vectorizer, X_train_np = vectorize_texts(x_train, MAX_FEATURES)
X_test_np = vectorizer.transform(x_test).toarray()
print(f"\n=== Antes da normalização ===")
print(f"X_train shape : {X_train_np.shape}")
print(f"Norma média   : {np.linalg.norm(X_train_np, axis=1).mean():.4f}")

X_train_np = normalize_features(X_train_np)
X_test_np  = normalize_features(X_test_np)
print(f"\n=== Após normalização L2 ===")
print(f"Norma média   : {np.linalg.norm(X_train_np, axis=1).mean():.4f}  ← deve ser ≈ 1.0")

=== Análise NumPy do dataset ===
Total samples  : 1597
Total columns  : 27
Rating min/max : 1 / 5
Rating mean    : 4.36
Rating std     : 1.02
  Rating 1: 42 reviews
  Rating 2: 34 reviews
  Rating 3: 124 reviews
  Rating 4: 236 reviews
  Rating 5: 741 reviews

Amostras após pré-processamento: 1053
Distribuição:
sentiment
1    977
0     76

=== Antes da normalização ===
X_train shape : (842, 2000)
Norma média   : 0.9976

=== Após normalização L2 ===
Norma média   : 0.9976  ← deve ser ≈ 1.0


---
## Entrega 3 — PyTorch Parte 1: Tensores, Dataset e DataLoader
> Etapa 5

In [ ]:
import torch
from src.training.train import to_tensors, create_dataloader
from src.utils.config import BATCH_SIZE

X_train_t, y_train_t = to_tensors(X_train_np, y_train.to_numpy())
X_test_t,  y_test_t  = to_tensors(X_test_np,  y_test.to_numpy())
train_loader = create_dataloader(X_train_t, y_train_t, BATCH_SIZE)
val_loader   = create_dataloader(X_test_t,  y_test_t,  BATCH_SIZE, shuffle=False)

print("=== Tensores PyTorch ===")
print(f"X_train : {X_train_t.shape} | {X_train_t.dtype}")
print(f"y_train : {y_train_t.shape} | {y_train_t.dtype}")
print(f"\n=== DataLoaders ===")
print(f"Batches treino    : {len(train_loader)}")
print(f"Batches validação : {len(val_loader)}")
print(f"Device            : {'cuda' if torch.cuda.is_available() else 'cpu'}")
Xb, yb = next(iter(train_loader))
print(f"\nBatch exemplo: X={Xb.shape}, y={yb.shape}")

---
## Entrega 4 — PyTorch Parte 2: Modelo, Treinamento e Inferência
> Etapa 6

In [ ]:
import torch.nn as nn
from src.models.model import build_model, save_model, load_model, predict
from src.utils.config import HIDDEN_DIM, OUTPUT_DIM, MODEL_PATH

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(MAX_FEATURES, HIDDEN_DIM, OUTPUT_DIM).to(device)
print("=== Arquitetura SentimentMLP ===")
print(model)
print(f"\nParâmetros: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from src.training.train import run_training
from src.utils.config import LEARNING_RATE, NUM_EPOCHS

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()
print("=== Loop de Treinamento + Validação ===")
model = run_training(model, train_loader, val_loader, optimizer, criterion, NUM_EPOCHS, device)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from src.evaluation.metrics import evaluate_model, print_report

y_pred = predict(model, X_test_t, device)
metrics = evaluate_model(y_test.to_numpy(), y_pred)
print("=== Métricas ===")
print_report(metrics)

ConfusionMatrixDisplay(confusion_matrix(y_test.to_numpy(), y_pred),
                       display_labels=["Negative", "Positive"]).plot(colorbar=False)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
save_model(model, MODEL_PATH)
model_reloaded = load_model(MODEL_PATH, MAX_FEATURES, HIDDEN_DIM, OUTPUT_DIM)
m2 = evaluate_model(y_test.to_numpy(), predict(model_reloaded, X_test_t, torch.device("cpu")))
print(f"Accuracy original : {metrics['accuracy']:.4f}")
print(f"Accuracy reloaded : {m2['accuracy']:.4f}  ← deve ser idêntica")

In [ ]:
from src.inference.predict import predict_batch

exemplos = [
    "This product is absolutely amazing, I love it!",
    "Terrible quality, broke after one day. Total waste of money.",
    "Best purchase I have ever made, highly recommend!",
    "Very disappointed, does not work at all.",
]
print("=== Inferência ===")
for texto, lbl in zip(exemplos, predict_batch(exemplos, model, vectorizer, device)):
    print(f"  [{'POSITIVE' if lbl else 'NEGATIVE'}] {texto}")

---
## Entrega 5 — Exercícios Intermediários
> Etapa 7

In [ ]:
import pandas as pd

configs = [
    {"name": "Baseline",   "hidden_dim": 256, "lr": 1e-3, "epochs": 10},
    {"name": "Rede maior", "hidden_dim": 512, "lr": 1e-3, "epochs": 10},
]
resultados = []
for cfg in configs:
    m = build_model(MAX_FEATURES, cfg["hidden_dim"], OUTPUT_DIM).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=cfg["lr"])
    run_training(m, train_loader, val_loader, opt, nn.BCEWithLogitsLoss(), cfg["epochs"], device)
    met = evaluate_model(y_test.to_numpy(), predict(m, X_test_t, device))
    resultados.append({"Config": cfg["name"], **{k: round(v, 4) for k, v in met.items()}})

df_res = pd.DataFrame(resultados).set_index("Config")
print("=== Tabela de Resultados ===")
display(df_res)

df_res[["accuracy", "f1_score"]].plot(kind="bar", figsize=(7, 4), ylim=(0.85, 1.0))
plt.title("Comparação: Accuracy e F1")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

diff = df_res.loc["Rede maior", "f1_score"] - df_res.loc["Baseline", "f1_score"]
print(f"\nΔF1 (Rede maior vs Baseline) = {diff:+.4f}")
print("Dataset pequeno: mais parâmetros não garante ganho expressivo.")

---
## Entrega 6 — Testes Automatizados com unittest
> Etapa 8

In [ ]:
import unittest
suite  = unittest.TestLoader().discover(start_dir="tests", pattern="test_*.py")
result = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\nTotal: {result.testsRun} | Falhas: {len(result.failures)} | Erros: {len(result.errors)}")

---
## Entrega 7 — Visão do Sistema, Requisitos e Arquitetura
> Etapas 9, 10 e 11

In [ ]:
from IPython.display import Markdown, display
for doc in ["docs/vision.md", "docs/requirements.md", "docs/architecture.md"]:
    print(f"\n{'='*60}\n  {doc}\n{'='*60}")
    with open(doc, encoding="utf-8") as f:
        display(Markdown(f.read()))

---
## Entrega 8 — Versionamento com Git
> Etapa 12

In [ ]:
import subprocess
branch  = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
log     = subprocess.check_output(["git", "log", "--oneline", "--graph", "-10"], text=True)
contrib = subprocess.check_output(["git", "shortlog", "-sn", "--all"], text=True)
print(f"Branch: {branch}\n")
print("=== Histórico ===")
print(log)
print("=== Contribuições ===")
print(contrib)

---
## Entrega Final — Pipeline Completo via main.py
> Etapa 13

In [ ]:
import subprocess, json

print("=== Executando main.py ===")
res = subprocess.run(["python", "main.py"], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr)

print("=== Artefatos gerados ===")
for path in ["data/processed/reviews_processed.csv",
             "results/models/model.pt",
             "results/models/vectorizer.pkl",
             "results/metrics/metrics.json",
             "results/figures/confusion_matrix.png"]:
    ok = os.path.exists(path)
    sz = f"{os.path.getsize(path):,} bytes" if ok else "—"
    print(f"  {'✅' if ok else '❌'} {path:<45} {sz}")

print("\n=== Métricas finais ===")
with open("results/metrics/metrics.json") as f:
    for k, v in json.load(f).items():
        print(f"  {k:<12}: {v:.4f}")